In [4]:
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support


In [ ]:
train_path = "/content/drive/MyDrive/Colab Notebooks/task_a_training_set_1.parquet"
valid_path = "/content/drive/MyDrive/Colab Notebooks/task_a_validation_set.parquet"

In [ ]:
class CodeParquetDataset(Dataset):
    def __init__(self, parquet_path, tokenizer, max_len=512, encoder_prefix="<encoder-only>"):
        self.df = pd.read_parquet(parquet_path)
        assert "code" in self.df.columns and "label" in self.df.columns, \
            "Parquet must contain 'code' and 'label' columns."
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.encoder_prefix = encoder_prefix

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        code = str(row["code"])
        text = f"{self.encoder_prefix} {code}"
        enc = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding=False,
            return_tensors=None
        )
        enc["labels"] = int(row["label"])
        return enc


In [6]:
import torch.nn as nn

class UniXcoderBinaryHead(nn.Module):
    def __init__(self, base_model_name="microsoft/unixcoder-base", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        last_hidden = outputs.last_hidden_state

        # Mean pooling with mask
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        summed = torch.sum(last_hidden * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-6)
        pooled = summed / denom

        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels.float())
        return {"loss": loss, "logits": logits}


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": f1, "precision": p, "recall": r}


In [ ]:
model_name = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder_prefix = "<encoder-only>"

# Add special token if needed
if encoder_prefix not in tokenizer.get_vocab():
    tokenizer.add_tokens([encoder_prefix], special_tokens=True)

train_dataset = CodeParquetDataset(train_path, tokenizer, encoder_prefix=encoder_prefix)
valid_dataset = CodeParquetDataset(valid_path, tokenizer, encoder_prefix=encoder_prefix)

model = UniXcoderBinaryHead(model_name)
if encoder_prefix in tokenizer.get_vocab():
    model.encoder.resize_token_embeddings(len(tokenizer))

collator = DataCollatorWithPadding(tokenizer)
set_seed(42)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./unixcoder_taskA_output",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    gradient_accumulation_steps=2,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    warmup_ratio=0.06,
    weight_decay=0.01,
    logging_steps=100,
    report_to="none",
    fp16=True,
)


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
trainer.save_model("./unixcoder_taskA_final")
tokenizer.save_pretrained("./unixcoder_taskA_final")

In [ ]:
preds = trainer.predict(valid_dataset)
logits = preds.predictions
labels = preds.label_ids
probs = 1 / (1 + np.exp(-logits))
pred_classes = (probs >= 0.5).astype(int)

f1 = f1_score(labels, pred_classes, average="macro")
print(f"Macro F1 = {f1:.4f}")


### PEFT Fine-Tuning with LoRA

In [ ]:
# ================================================
# 🧠 Fine-tune UniXcoder with LoRA (Binary Classification)
# ================================================

# !pip install -q torch==2.8.0+cu126 torchvision==0.23.0+cu126 torchaudio==2.8.0+cu126 --index-url https://download.pytorch.org/whl/cu126
# !pip install -q transformers==4.44.2 accelerate==0.33.0 datasets==3.0.1 evaluate==0.4.2 scikit-learn==1.6.0 pandas pyarrow peft==0.13.0 bitsandbytes==0.44.1

# -----------------------------
# 📦 Imports
# -----------------------------
import torch
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModel, Trainer, TrainingArguments,
    DataCollatorWithPadding, set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from peft import LoraConfig, get_peft_model, TaskType

# -----------------------------
# ⚙️ Dataset Class
# -----------------------------
class CodeParquetDataset(Dataset):
    def __init__(self, parquet_path, tokenizer, max_len=512, encoder_prefix="<encoder-only>"):
        self.df = pd.read_parquet(parquet_path)
        assert "code" in self.df.columns and "label" in self.df.columns, \
            "Parquet file must contain 'code' and 'label' columns"
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.encoder_prefix = encoder_prefix

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = f"{self.encoder_prefix} {row['code']}"
        enc = self.tokenizer(text, truncation=True, max_length=self.max_len, padding=False)
        enc["labels"] = int(row["label"])
        return enc

# -----------------------------
# 🧱 Model with Classification Head
# -----------------------------
class UniXcoderBinaryHead(nn.Module):
    def __init__(self, base_model_name="microsoft/unixcoder-base", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.config = self.encoder.config              # needed by PEFT
        hidden = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    # ⬇️ accept **kwargs so Trainer/PEFT can pass inputs_embeds, token_type_ids, etc.
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        # forward *all* remaining args to the base model
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )
        last_hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
        pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)

        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss()(logits, labels.float())
        return {"loss": loss, "logits": logits}

# -----------------------------
# 📊 Evaluation Metrics
# -----------------------------
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {"accuracy": acc, "macro_f1": f1, "precision": p, "recall": r}

# -----------------------------
# 💾 Load Tokenizer and Data
# -----------------------------
# train_path = "/content/drive/MyDrive/SemEval2026/train.parquet"
# valid_path = "/content/drive/MyDrive/SemEval2026/valid.parquet"

model_name = "microsoft/unixcoder-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder_prefix = "<encoder-only>"
if encoder_prefix not in tokenizer.get_vocab():
    tokenizer.add_tokens([encoder_prefix], special_tokens=True)

train_dataset = CodeParquetDataset(train_path, tokenizer, encoder_prefix=encoder_prefix)
valid_dataset = CodeParquetDataset(valid_path, tokenizer, encoder_prefix=encoder_prefix)

# -----------------------------
# 🧩 Initialize Model + LoRA
# -----------------------------
model = UniXcoderBinaryHead(model_name)
model.encoder.resize_token_embeddings(len(tokenizer))

# Attach LoRA adapters
peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["query", "value"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# -----------------------------
# ⚡ Training Configuration
# -----------------------------
collator = DataCollatorWithPadding(tokenizer)
set_seed(42)
# training_args = TrainingArguments(
#     output_dir="./unixcoder_lora_output",
#     learning_rate=2e-4,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=16,
#     num_train_epochs=2,
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     load_best_model_at_end=True,
#     metric_for_best_model="macro_f1",
#     greater_is_better=True,
#     fp16=True,
#     logging_steps=100,
#     report_to="none"
# )

training_args = TrainingArguments(
    output_dir="./unixcoder_lora_output",
    learning_rate=3e-4,
    per_device_train_batch_size=8,     # smaller batch
    gradient_accumulation_steps=2,     # keeps effective batch size same
    num_train_epochs=3,                # shorter for quick testing
    max_steps=10000,                   # limit steps for debug
    evaluation_strategy="steps",
    eval_steps=1000,
    fp16=True,
    report_to="none"
)

# -----------------------------
# 🚀 Train
# -----------------------------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)
trainer.train()

# -----------------------------
# ✅ Evaluate and Save
# -----------------------------
results = trainer.evaluate()
print("\n✅ Validation Results:", results)

model.save_pretrained("./unixcoder_lora_taskA_final")
tokenizer.save_pretrained("./unixcoder_lora_taskA_final")
print("\n✅ Model saved at ./unixcoder_lora_taskA_final")


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


trainable params: 295,681 || all params: 126,226,178 || trainable%: 0.2342


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Accuracy,Macro F1,Precision,Recall
1000,0.068500,0.048250,0.985750,0.985717,0.985805,0.985636
2000,0.049600,0.046836,0.986410,0.986380,0.986435,0.986327
3000,0.040400,0.037966,0.989820,0.989797,0.989851,0.989746
4000,0.040200,0.033791,0.990910,0.990889,0.990964,0.990819


KeyboardInterrupt: 

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

!cp -r ./unixcoder_lora_output/checkpoint-4000 /content/drive/MyDrive/unixcoder_lora_checkpoint_4000


## Reloading Model for Evaluation

In [7]:
from transformers import AutoModel, AutoTokenizer
from peft import PeftModel, PeftConfig

checkpoint_path = "./unixcoder_lora_output/checkpoint-4000"
base_model_name = "microsoft/unixcoder-base"  # ✅ explicitly define base model

# Load base UniXcoder model
base_model = AutoModel.from_pretrained(base_model_name)

# Load LoRA adapter into the base model
model = PeftModel.from_pretrained(base_model, checkpoint_path).to("cuda")
model.eval()

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
encoder_prefix = "<encoder-only>"


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

ValueError: Can't find 'adapter_config.json' at './unixcoder_lora_output/checkpoint-4000'

In [ ]:
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")
val_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_validation_set.parquet")


In [ ]:
# def evaluate_model(df, split_name="validation", threshold=0.5, batch_size=8):
#     preds, labels = [], []
#     model.eval()

#     for i in tqdm(range(0, len(df), batch_size), desc=f"Evaluating {split_name}"):
#         batch = df.iloc[i:i+batch_size]
#         texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

#         inputs = tokenizer(
#             texts,
#             return_tensors="pt",
#             truncation=True,
#             max_length=512,
#             padding=True
#         ).to("cuda")

#         with torch.no_grad():
#             outputs = model(**inputs)
#             if hasattr(outputs, "logits"):
#                 logits = outputs.logits
#             else:
#                 logits = outputs.last_hidden_state.mean(dim=1)
#             if logits.ndim > 1:
#                 logits = logits.mean(dim=-1)
#             probs = torch.sigmoid(logits).cpu().numpy()

#         preds.extend((probs >= threshold).astype(int).tolist())
#         labels.extend(batch["label"].tolist())

#     acc = accuracy_score(labels, preds)
#     p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)

#     print(f"\n📊 {split_name.upper()} RESULTS:")
#     print(f"Accuracy:  {acc:.4f}")
#     print(f"Precision: {p:.4f}")
#     print(f"Recall:    {r:.4f}")
#     print(f"Macro F1:  {f1:.4f}")

#     return preds, {"accuracy": acc, "precision": p, "recall": r, "macro_f1": f1}
# val_preds, val_metrics = evaluate_model(val_df, "validation")
# test_preds, test_metrics = evaluate_model(test_df, "test")


In [ ]:
test_df.head()

,code,generator,label,language
0,public Vector To(Vector o)\n {\n ...,Human,0,C#
1,func (v *DefaultMessageSyntaxValidator) Valida...,Human,0,Go
2,"""""""Module managing testsuite capabilities\n\nC...",Human,0,Python
3,void Anvil::Image::on_memory_backing_opaque_up...,Human,0,C++
4,bool NOMAD::Priority_Eval_Point::dominates\n( ...,Human,0,C++


In [9]:
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# ==============================================================
# 🧱 Define the same architecture used during training
# ==============================================================

class UniXcoderBinaryHead(nn.Module):
    def __init__(self, base_model_name="microsoft/unixcoder-base", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.config = self.encoder.config
        hidden = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).type_as(hidden)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits


# ==============================================================
# ⚙️ Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-4000"
base_model_name = "microsoft/unixcoder-base"

model = UniXcoderBinaryHead(base_model_name)
model = PeftModel.from_pretrained(model, checkpoint_path)
model.to("cuda").eval()

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
encoder_prefix = "<encoder-only>"

# ==============================================================
# 📂 Load your test dataset
# ==============================================================

print(f"Loaded {len(test_df)} samples.")

# ==============================================================
# 🚀 Inference Loop
# ==============================================================

preds = []
batch_size = 8

for i in tqdm(range(0, len(test_df), batch_size), desc="Predicting"):
    batch = test_df.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        logits = model(**inputs)
        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)

# ==============================================================
# 💾 Create Submission File
# ==============================================================

submission = pd.DataFrame({
    "ID": test_df.index,  # or test_df["id"] if it exists
    "label": preds
})
submission.to_csv("submission_taskA.csv", index=False)

print("✅ submission_taskA.csv saved successfully!")
submission.head()


In [ ]:
test_df.tail()

,code,generator,label,language
995,/**-------------------------------------------...,Human,0,C
996,"void AuxFunc(const dTensor1& xpts, \n\t dT...",Human,0,C++
997,def plot_confusion_matrix(\n context: MLCli...,Human,0,Python
998,import sys\nfrom collections import Counter\n\...,human,0,Python
999,"def assess_dimension(spectrum, rank, n_samples...",Human,0,Python


In [ ]:
len(preds)

1000

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report
import numpy as np

# Convert predictions and labels to numpy arrays
y_true = test_df["label"].to_numpy()
y_pred = np.array(preds)

# Compute all relevant metrics
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)

print("\n📊 Evaluation Results (Binary Classification)")
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"Macro F1:   {f1:.4f}")

# (Optional) show detailed per-class report
print("\nDetailed Classification Report:\n")
print(classification_report(y_true, y_pred, digits=4))



📊 Evaluation Results (Binary Classification)
Accuracy:   0.2450
Precision:  0.3981
Recall:     0.4406
Macro F1:   0.2359

Detailed Classification Report:

              precision    recall  f1-score   support

           0     0.5965    0.0875    0.1526       777
           1     0.1998    0.7937    0.3192       223

    accuracy                         0.2450      1000
   macro avg     0.3981    0.4406    0.2359      1000
weighted avg     0.5080    0.2450    0.1898      1000



In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# ==============================================================
# 🧱 Define the same architecture used during training
# ==============================================================

class UniXcoderBinaryHead(nn.Module):
    def __init__(self, base_model_name="microsoft/unixcoder-base", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.config = self.encoder.config
        hidden = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).type_as(hidden)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits


# ==============================================================
# ⚙️ Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-4000"
base_model_name = "microsoft/unixcoder-base"

model = UniXcoderBinaryHead(base_model_name)
model = PeftModel.from_pretrained(model, checkpoint_path)
model.to("cuda").eval()

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
encoder_prefix = "<encoder-only>"

# ==============================================================
# 📂 Load your test dataset
# ==============================================================


# ==============================================================
# 🚀 Inference Loop
# ==============================================================

preds = []
batch_size = 8

for i in tqdm(range(0, len(val_df), batch_size), desc="Predicting"):
    batch = val_df.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        logits = model(**inputs)
        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)

# ==============================================================
# 💾 Create Submission File
# ==============================================================

submission = pd.DataFrame({
    "ID": val_df.index,  # or test_df["id"] if it exists
    "label": preds
})
submission.to_csv("submission_taskA.csv", index=False)

print("✅ submission_val.csv saved successfully!")
submission.head()


Predicting:   0%|          | 0/12500 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel
from peft import PeftModel

# ==============================================================
# 🧱 Define the same architecture used during training
# ==============================================================

class UniXcoderBinaryHead(nn.Module):
    def __init__(self, base_model_name="microsoft/unixcoder-base", dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        self.config = self.encoder.config
        hidden = self.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, 1)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        hidden = outputs.last_hidden_state
        mask = attention_mask.unsqueeze(-1).type_as(hidden)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        logits = self.classifier(self.dropout(pooled)).squeeze(-1)
        return logits


# ==============================================================
# ⚙️ Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-4000"
base_model_name = "microsoft/unixcoder-base"

# 🔧 FIX 1: Load tokenizer from checkpoint (has custom tokens!)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
encoder_prefix = "<encoder-only>"

# Initialize base model
model = UniXcoderBinaryHead(base_model_name)

# 🔧 FIX 2: Resize embeddings to match tokenizer
model.encoder.resize_token_embeddings(len(tokenizer))

# Load LoRA adapter
model = PeftModel.from_pretrained(model, checkpoint_path)

# Set to eval mode and move to GPU
model.eval()
model.to("cuda")

print(f"✅ Model loaded successfully")
print(f"📊 Tokenizer vocab size: {len(tokenizer)}")
print(f"🔤 Special token '{encoder_prefix}' in vocab: {encoder_prefix in tokenizer.get_vocab()}")

# ==============================================================
# 📂 Load your test dataset
# ==============================================================

test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")

# ==============================================================
# 🚀 Inference Loop
# ==============================================================

preds = []
probs_list = []
batch_size = 8

for i in tqdm(range(0, len(test_df), batch_size), desc="Predicting"):
    batch = test_df.iloc[i:i+batch_size]

    # Add encoder prefix to each code sample
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    # Tokenize
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    # Inference
    with torch.no_grad():
        logits = model(**inputs)
        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)
    probs_list.extend(probs.tolist())

# ==============================================================
# 💾 Create Submission File
# ==============================================================

submission = pd.DataFrame({
    "ID": test_df.index,  # or test_df["id"] if it exists
    "label": preds
})
submission.to_csv("submission_taskA.csv", index=False)

print(f"\n✅ Submission saved successfully!")
print(f"📊 Prediction distribution:")
print(f"   Class 0: {sum(p == 0 for p in preds)} ({sum(p == 0 for p in preds)/len(preds)*100:.1f}%)")
print(f"   Class 1: {sum(p == 1 for p in preds)} ({sum(p == 1 for p in preds)/len(preds)*100:.1f}%)")
print(f"   Average probability: {sum(probs_list)/len(probs_list):.4f}")

submission.head(10)

✅ Model loaded successfully
📊 Tokenizer vocab size: 51416
🔤 Special token '<encoder-only>' in vocab: True


Predicting:   0%|          | 0/125 [00:00<?, ?it/s]

TypeError: RobertaModel.forward() got an unexpected keyword argument 'labels'

In [ ]:
# Load the test dataset again
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")

# Recreate test_df_inference (same logic as before)
if 'label' in test_df.columns:
    true_labels = test_df['label'].tolist()  # Save for evaluation
    test_df_inference = test_df.drop(columns=['label'])  # Remove labels for inference
    print("✅ Labels saved separately for evaluation")
    has_labels = True
else:
    true_labels = None
    test_df_inference = test_df.copy()
    has_labels = False
    print("⚠️  No labels found in test data")

print(f"📊 Test samples: {len(test_df_inference)}")
print(f"📋 Columns: {test_df_inference.columns.tolist()}")

In [ ]:
# ==============================================================
# ⚙️ CORRECTED: Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-4000"
base_model_name = "microsoft/unixcoder-base"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
encoder_prefix = "<encoder-only>"

# Initialize base model
model = UniXcoderBinaryHead(base_model_name)
model.encoder.resize_token_embeddings(len(tokenizer))

# 🔧 FIX: Wrap only the encoder with PEFT, not the whole model
model.encoder = PeftModel.from_pretrained(model.encoder, checkpoint_path)

# Merge adapter weights for faster inference (optional but recommended)
model.encoder = model.encoder.merge_and_unload()

model.eval()
model.to("cuda")

# ==============================================================
# 🚀 CORRECTED: Inference Loop
# ==============================================================

preds = []
probs_list = []
batch_size = 8

for i in tqdm(range(0, len(test_df_inference), batch_size), desc="Predicting"):
    batch = test_df_inference.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        # 🔧 FIX: Handle dict output
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

        # Extract logits from dict if needed
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs

        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)
    probs_list.extend(probs.tolist())

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.encoder.layer.0.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.2.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.2.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.2.attention.self.value.lora_A.default.weigh

Predicting:   0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
# # Method 1: Using threshold of 0.5 (standard for binary classification)
# preds = (np.array(probs_list) >= 0.5).astype(int).tolist()

# # Method 2: If you already have probs_list as a list
# preds = [1 if prob >= 0.5 else 0 for prob in probs_list]

# # Method 3: Using numpy for efficiency
# preds = np.where(np.array(probs_list) >= 0.5, 1, 0).tolist()

import numpy as np
import pandas as pd

# Convert probabilities to labels
preds = (np.array(probs_list) >= 0.5).astype(int)

# Create submission file
submission = pd.DataFrame({
    "ID": test_df.index,  # or range(len(test_df)) if no index
    "label": preds,
    "probability": probs_list
})

# Save submission
submission.to_csv("submission_taskA.csv", index=False)
print(f"✅ Submission saved with {len(submission)} predictions")
print(f"\nLabel distribution:")
print(f"  Class 0: {sum(preds == 0)} ({sum(preds == 0)/len(preds)*100:.1f}%)")
print(f"  Class 1: {sum(preds == 1)} ({sum(preds == 1)/len(preds)*100:.1f}%)")

✅ Submission saved with 1000 predictions

Label distribution:
  Class 0: 921 (92.1%)
  Class 1: 79 (7.9%)


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# -----------------------------
# Load files
# -----------------------------
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")
pred_df = pd.read_csv("submission_taskA.csv")

# Ensure they’re aligned (if your CSV has 'id' column)
if "id" in pred_df.columns and "id" in test_df.columns:
    merged = test_df.merge(pred_df, on="id", suffixes=("_true", "_pred"))
    y_true = merged["label_true"]
    y_pred = merged["label_pred"]
else:
    # If both are in the same order
    y_true = test_df["label"]
    y_pred = pred_df["label"]

# -----------------------------
# Compute metrics
# -----------------------------
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
report = classification_report(y_true, y_pred, digits=4)

# -----------------------------
# Print results
# -----------------------------
print("📊 Evaluation Results (Binary Classification)")
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"Macro F1:   {f1:.4f}")
print("\nDetailed Classification Report:\n")
print(report)


📊 Evaluation Results (Binary Classification)
Accuracy:   0.7200
Precision:  0.4545
Recall:     0.4809
Macro F1:   0.4540

Detailed Classification Report:

              precision    recall  f1-score   support

           0     0.7698    0.9125    0.8351       777
           1     0.1392    0.0493    0.0728       223

    accuracy                         0.7200      1000
   macro avg     0.4545    0.4809    0.4540      1000
weighted avg     0.6292    0.7200    0.6651      1000



## Evaluation with 3500 Checkpoint


In [ ]:
# ==============================================================
# ⚙️ CORRECTED: Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-3500"
base_model_name = "microsoft/unixcoder-base"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
encoder_prefix = "<encoder-only>"

# Initialize base model
model = UniXcoderBinaryHead(base_model_name)
model.encoder.resize_token_embeddings(len(tokenizer))

# 🔧 FIX: Wrap only the encoder with PEFT, not the whole model
model.encoder = PeftModel.from_pretrained(model.encoder, checkpoint_path)

# Merge adapter weights for faster inference (optional but recommended)
model.encoder = model.encoder.merge_and_unload()

model.eval()
model.to("cuda")

# ==============================================================
# 🚀 CORRECTED: Inference Loop
# ==============================================================

preds = []
probs_list = []
batch_size = 8

for i in tqdm(range(0, len(test_df_inference), batch_size), desc="Predicting"):
    batch = test_df_inference.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        # 🔧 FIX: Handle dict output
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

        # Extract logits from dict if needed
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs

        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)
    probs_list.extend(probs.tolist())

Predicting:   0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
# # Method 1: Using threshold of 0.5 (standard for binary classification)
# preds = (np.array(probs_list) >= 0.5).astype(int).tolist()

# # Method 2: If you already have probs_list as a list
# preds = [1 if prob >= 0.5 else 0 for prob in probs_list]

# # Method 3: Using numpy for efficiency
# preds = np.where(np.array(probs_list) >= 0.5, 1, 0).tolist()

import numpy as np
import pandas as pd

# Convert probabilities to labels
preds = (np.array(probs_list) >= 0.5).astype(int)

# Create submission file
submission = pd.DataFrame({
    "ID": test_df.index,  # or range(len(test_df)) if no index
    "label": preds,
})

# Save submission
submission.to_csv("submission_3500.csv", index=False)
print(f"✅ Submission saved with {len(submission)} predictions")
print(f"\nLabel distribution:")
print(f"  Class 0: {sum(preds == 0)} ({sum(preds == 0)/len(preds)*100:.1f}%)")
print(f"  Class 1: {sum(preds == 1)} ({sum(preds == 1)/len(preds)*100:.1f}%)")

✅ Submission saved with 1000 predictions

Label distribution:
  Class 0: 741 (74.1%)
  Class 1: 259 (25.9%)


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# -----------------------------
# Load files
# -----------------------------
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")
pred_df = pd.read_csv("submission_3500.csv")

# Ensure they’re aligned (if your CSV has 'id' column)
if "id" in pred_df.columns and "id" in test_df.columns:
    merged = test_df.merge(pred_df, on="id", suffixes=("_true", "_pred"))
    y_true = merged["label_true"]
    y_pred = merged["label_pred"]
else:
    # If both are in the same order
    y_true = test_df["label"]
    y_pred = pred_df["label"]

# -----------------------------
# Compute metrics
# -----------------------------
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
report = classification_report(y_true, y_pred, digits=4)

# -----------------------------
# Print results
# -----------------------------
print("📊 Evaluation Results (Binary Classification)")
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"Macro F1:   {f1:.4f}")
print("\nDetailed Classification Report:\n")
print(report)


📊 Evaluation Results (Binary Classification)
Accuracy:   0.6580
Precision:  0.5319
Recall:     0.5353
Macro F1:   0.5326

Detailed Classification Report:

              precision    recall  f1-score   support

           0     0.7935    0.7568    0.7747       777
           1     0.2703    0.3139    0.2905       223

    accuracy                         0.6580      1000
   macro avg     0.5319    0.5353    0.5326      1000
weighted avg     0.6768    0.6580    0.6667      1000



In [ ]:
!cp -r ./unixcoder_lora_output/checkpoint-3500 /content/drive/MyDrive/unixcoder_lora_checkpoint_3500

## Reloading for Evaluation on test.parquet

In [ ]:
# Load the test dataset again
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/test.parquet")

# Recreate test_df_inference (same logic as before)
if 'label' in test_df.columns:
    true_labels = test_df['label'].tolist()  # Save for evaluation
    test_df_inference = test_df.drop(columns=['label'])  # Remove labels for inference
    print("✅ Labels saved separately for evaluation")
    has_labels = True
else:
    true_labels = None
    test_df_inference = test_df.copy()
    has_labels = False
    print("⚠️  No labels found in test data")

print(f"📊 Test samples: {len(test_df_inference)}")
print(f"📋 Columns: {test_df_inference.columns.tolist()}")

⚠️  No labels found in test data
📊 Test samples: 1000
📋 Columns: ['ID', 'code']


In [ ]:
# ==============================================================
# ⚙️ CORRECTED: Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-3500"
base_model_name = "microsoft/unixcoder-base"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
encoder_prefix = "<encoder-only>"

# Initialize base model
model = UniXcoderBinaryHead(base_model_name)
model.encoder.resize_token_embeddings(len(tokenizer))

# 🔧 FIX: Wrap only the encoder with PEFT, not the whole model
model.encoder = PeftModel.from_pretrained(model.encoder, checkpoint_path)

# Merge adapter weights for faster inference (optional but recommended)
model.encoder = model.encoder.merge_and_unload()

model.eval()
model.to("cuda")

# ==============================================================
# 🚀 CORRECTED: Inference Loop
# ==============================================================

preds = []
probs_list = []
batch_size = 8

for i in tqdm(range(0, len(test_df_inference), batch_size), desc="Predicting"):
    batch = test_df_inference.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        # 🔧 FIX: Handle dict output
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

        # Extract logits from dict if needed
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs

        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)
    probs_list.extend(probs.tolist())

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.encoder.layer.0.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.2.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.2.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.2.attention.self.value.lora_A.default.weigh

Predicting:   0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
# # Method 1: Using threshold of 0.5 (standard for binary classification)
# preds = (np.array(probs_list) >= 0.5).astype(int).tolist()

# # Method 2: If you already have probs_list as a list
# preds = [1 if prob >= 0.5 else 0 for prob in probs_list]

# # Method 3: Using numpy for efficiency
# preds = np.where(np.array(probs_list) >= 0.5, 1, 0).tolist()

import numpy as np
import pandas as pd

# Convert probabilities to labels
preds = (np.array(probs_list) >= 0.5).astype(int)

# Create submission file
submission = pd.DataFrame({
    "ID": test_df.index,  # or range(len(test_df)) if no index
    "label": preds,
})

# Save submission
submission.to_csv("submission_3000_test.csv", index=False)
print(f"✅ Submission saved with {len(submission)} predictions")
print(f"\nLabel distribution:")
print(f"  Class 0: {sum(preds == 0)} ({sum(preds == 0)/len(preds)*100:.1f}%)")
print(f"  Class 1: {sum(preds == 1)} ({sum(preds == 1)/len(preds)*100:.1f}%)")

✅ Submission saved with 1000 predictions

Label distribution:
  Class 0: 753 (75.3%)
  Class 1: 247 (24.7%)


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# -----------------------------
# Load files
# -----------------------------
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")
pred_df = pd.read_csv("submission_3000_test.csv")

# Ensure they’re aligned (if your CSV has 'id' column)
if "id" in pred_df.columns and "id" in test_df.columns:
    merged = test_df.merge(pred_df, on="id", suffixes=("_true", "_pred"))
    y_true = merged["label_true"]
    y_pred = merged["label_pred"]
else:
    # If both are in the same order
    y_true = test_df["label"]
    y_pred = pred_df["label"]

# -----------------------------
# Compute metrics
# -----------------------------
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
report = classification_report(y_true, y_pred, digits=4)

# -----------------------------
# Print results
# -----------------------------
print("📊 Evaluation Results (Binary Classification)")
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"Macro F1:   {f1:.4f}")
print("\nDetailed Classification Report:\n")
print(report)


📊 Evaluation Results (Binary Classification)
Accuracy:   0.6340
Precision:  0.4917
Recall:     0.4911
Macro F1:   0.4910

Detailed Classification Report:

              precision    recall  f1-score   support

           0     0.7729    0.7490    0.7608       777
           1     0.2105    0.2332    0.2213       223

    accuracy                         0.6340      1000
   macro avg     0.4917    0.4911    0.4910      1000
weighted avg     0.6475    0.6340    0.6405      1000



,ID,code
2005,2005,"A = map(int, input().split())\na = sorted(A)\n..."
2384,2384,a = list(input())\nb = list(input())\nif a[0] ...
3526,3526,"(n, d) = list(map(int, input().split(' ')))\nx..."
3926,3926,def check8(num):\n\tnum = str(num)\n\tfor i in...
7222,7222,0\n import sys\n def bfs(in):\n arr = list(ma...
...,...,...
1102578,1102578,﻿using AutoMapper;\nusing MedMan.Application.A...
1102766,1102766,internal void AddAllocatedEvaluator(IAllocated...
1103412,1103412,"List<T> GetObjectsAround<T>(string tag, float ..."
1104673,1104673,static void\ndefault_noticereporter(const char...


## Reloading for Evaluation on test.parquet checkpoint-4000

In [ ]:
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/test.parquet")

# Load the test dataset again
# valid_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_validation_set.parquet")

# Recreate test_df_inference (same logic as before)
if 'label' in test_df.columns:
    true_labels = test_df['label'].tolist()  # Save for evaluation
    test_df_inference = test_df.drop(columns=['label'])  # Remove labels for inference
    print("✅ Labels saved separately for evaluation")
    has_labels = True
else:
    true_labels = None
    test_df_inference = test_df.copy()
    has_labels = False
    print("⚠️  No labels found in test data")

print(f"📊 Test samples: {len(test_df_inference)}")
print(f"📋 Columns: {test_df_inference.columns.tolist()}")

⚠️  No labels found in test data
📊 Test samples: 1000
📋 Columns: ['ID', 'code']


In [ ]:
# ==============================================================
# ⚙️ CORRECTED: Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/unixcoder_lora_output/checkpoint-4000"
base_model_name = "microsoft/unixcoder-base"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
encoder_prefix = "<encoder-only>"

# Initialize base model
model = UniXcoderBinaryHead(base_model_name)
model.encoder.resize_token_embeddings(len(tokenizer))

# 🔧 FIX: Wrap only the encoder with PEFT, not the whole model
model.encoder = PeftModel.from_pretrained(model.encoder, checkpoint_path)

# Merge adapter weights for faster inference (optional but recommended)
model.encoder = model.encoder.merge_and_unload()

model.eval()
model.to("cuda")

# ==============================================================
# 🚀 CORRECTED: Inference Loop
# ==============================================================

preds = []
probs_list = []
batch_size = 8

for i in tqdm(range(0, len(test_df_inference), batch_size), desc="Predicting"):
    batch = test_df_inference.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        # 🔧 FIX: Handle dict output
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

        # Extract logits from dict if needed
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs

        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)
    probs_list.extend(probs.tolist())

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.encoder.layer.0.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.0.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.1.attention.self.value.lora_A.default.weight', 'base_model.model.encoder.layer.1.attention.self.value.lora_B.default.weight', 'base_model.model.encoder.layer.2.attention.self.query.lora_A.default.weight', 'base_model.model.encoder.layer.2.attention.self.query.lora_B.default.weight', 'base_model.model.encoder.layer.2.attention.self.value.lora_A.default.weigh

Predicting:   0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
# # Method 1: Using threshold of 0.5 (standard for binary classification)
# preds = (np.array(probs_list) >= 0.5).astype(int).tolist()

# # Method 2: If you already have probs_list as a list
# preds = [1 if prob >= 0.5 else 0 for prob in probs_list]

# # Method 3: Using numpy for efficiency
# preds = np.where(np.array(probs_list) >= 0.5, 1, 0).tolist()

import numpy as np
import pandas as pd

# Convert probabilities to labels
preds = (np.array(probs_list) >= 0.5).astype(int)

# Create submission file
submission = pd.DataFrame({
    "ID": test_df.index,  # or range(len(test_df)) if no index
    "label": preds,
})

# Save submission
submission.to_csv("submission_4000_test.csv", index=False)
print(f"✅ Submission saved with {len(submission)} predictions")
print(f"\nLabel distribution:")
print(f"  Class 0: {sum(preds == 0)} ({sum(preds == 0)/len(preds)*100:.1f}%)")
print(f"  Class 1: {sum(preds == 1)} ({sum(preds == 1)/len(preds)*100:.1f}%)")

✅ Submission saved with 1000 predictions

Label distribution:
  Class 0: 610 (61.0%)
  Class 1: 390 (39.0%)


In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

# -----------------------------
# Load files
# -----------------------------
test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/test_sample.parquet")
pred_df = pd.read_csv("submission_4000_test.csv")

# Ensure they’re aligned (if your CSV has 'id' column)
if "id" in pred_df.columns and "id" in test_df.columns:
    merged = test_df.merge(pred_df, on="id", suffixes=("_true", "_pred"))
    y_true = merged["label_true"]
    y_pred = merged["label_pred"]
else:
    # If both are in the same order
    y_true = test_df["label"]
    y_pred = pred_df["label"]

# -----------------------------
# Compute metrics
# -----------------------------
acc = accuracy_score(y_true, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
report = classification_report(y_true, y_pred, digits=4)

# -----------------------------
# Print results
# -----------------------------
print("📊 Evaluation Results (Binary Classification)")
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"Macro F1:   {f1:.4f}")
print("\nDetailed Classification Report:\n")
print(report)


📊 Evaluation Results (Binary Classification)
Accuracy:   0.5570
Precision:  0.4959
Recall:     0.4943
Macro F1:   0.4790

Detailed Classification Report:

              precision    recall  f1-score   support

           0     0.7738    0.6075    0.6806       777
           1     0.2179    0.3812    0.2773       223

    accuracy                         0.5570      1000
   macro avg     0.4959    0.4943    0.4790      1000
weighted avg     0.6498    0.5570    0.5907      1000



In [ ]:
test = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/test.parquet")
test.head()

,ID,code
2005,2005,"A = map(int, input().split())\na = sorted(A)\n..."
2384,2384,a = list(input())\nb = list(input())\nif a[0] ...
3526,3526,"(n, d) = list(map(int, input().split(' ')))\nx..."
3926,3926,def check8(num):\n\tnum = str(num)\n\tfor i in...
7222,7222,0\n import sys\n def bfs(in):\n arr = list(ma...


In [ ]:
test_sample = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/test_sample.parquet")
test_sample.head()

,code,generator,label,language
0,public Vector To(Vector o)\n {\n ...,Human,0,C#
1,func (v *DefaultMessageSyntaxValidator) Valida...,Human,0,Go
2,"""""""Module managing testsuite capabilities\n\nC...",Human,0,Python
3,void Anvil::Image::on_memory_backing_opaque_up...,Human,0,C++
4,bool NOMAD::Priority_Eval_Point::dominates\n( ...,Human,0,C++


In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 2005 to 1105460
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   ID      1000 non-null   int64 
 1   code    1000 non-null   object
dtypes: int64(1), object(1)
memory usage: 23.4+ KB


In [ ]:
test_sample = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_test_set_sample.parquet")
test_sample.head()


,code,generator,label,language
0,public Vector To(Vector o)\n {\n ...,Human,0,C#
1,func (v *DefaultMessageSyntaxValidator) Valida...,Human,0,Go
2,"""""""Module managing testsuite capabilities\n\nC...",Human,0,Python
3,void Anvil::Image::on_memory_backing_opaque_up...,Human,0,C++
4,bool NOMAD::Priority_Eval_Point::dominates\n( ...,Human,0,C++


In [ ]:
from transformers import AutoModel
model = AutoModel.from_pretrained("microsoft/unixcoder-base")

for name, _ in model.named_modules():
    if "attn" in name or "attention" in name:
        print(name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

encoder.layer.0.attention
encoder.layer.0.attention.self
encoder.layer.0.attention.self.query
encoder.layer.0.attention.self.key
encoder.layer.0.attention.self.value
encoder.layer.0.attention.self.dropout
encoder.layer.0.attention.output
encoder.layer.0.attention.output.dense
encoder.layer.0.attention.output.LayerNorm
encoder.layer.0.attention.output.dropout
encoder.layer.1.attention
encoder.layer.1.attention.self
encoder.layer.1.attention.self.query
encoder.layer.1.attention.self.key
encoder.layer.1.attention.self.value
encoder.layer.1.attention.self.dropout
encoder.layer.1.attention.output
encoder.layer.1.attention.output.dense
encoder.layer.1.attention.output.LayerNorm
encoder.layer.1.attention.output.dropout
encoder.layer.2.attention
encoder.layer.2.attention.self
encoder.layer.2.attention.self.query
encoder.layer.2.attention.self.key
encoder.layer.2.attention.self.value
encoder.layer.2.attention.self.dropout
encoder.layer.2.attention.output
encoder.layer.2.attention.output.dense
e

## Evaluation on Validation Data

In [2]:
import pandas as pd
# test_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/test.parquet")

# Load the valid dataset again
valid_df = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/task_a_validation_set.parquet")

# Recreate valid_df_inference (same logic as before)
if 'label' in valid_df.columns:
    true_labels = valid_df['label'].tolist()  # Save for evaluation
    valid_df_inference = valid_df.drop(columns=['label'])  # Remove labels for inference
    print("✅ Labels saved separately for evaluation")
    has_labels = True
else:
    true_labels = None
    valid_df_inference = valid_df.copy()
    has_labels = False
    print("⚠️  No labels found in valid data")

print(f"📊 Valid samples: {len(valid_df_inference)}")
print(f"📋 Columns: {valid_df_inference.columns.tolist()}")

✅ Labels saved separately for evaluation
📊 Valid samples: 100000
📋 Columns: ['code', 'generator', 'language']


In [13]:
# ==============================================================
# ⚙️ CORRECTED: Load model + adapter checkpoint
# ==============================================================

checkpoint_path = "/content/drive/MyDrive/unixcoder_lora_checkpoint_4000"
base_model_name = "microsoft/unixcoder-base"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)
encoder_prefix = "<encoder-only>"

# Initialize base model
model = UniXcoderBinaryHead(base_model_name)
model.encoder.resize_token_embeddings(len(tokenizer))

# 🔧 FIX: Wrap only the encoder with PEFT, not the whole model
model.encoder = PeftModel.from_pretrained(model.encoder, checkpoint_path)

# Merge adapter weights for faster inference (optional but recommended)
model.encoder = model.encoder.merge_and_unload()

model.eval()
model.to("cpu")

# ==============================================================
# 🚀 CORRECTED: Inference Loop
# ==============================================================

preds = []
probs_list = []
batch_size = 8

for i in tqdm(range(0, len(valid_df_inference), batch_size), desc="Predicting"):
    batch = valid_df_inference.iloc[i:i+batch_size]
    texts = [f"{encoder_prefix} {c}" for c in batch["code"]]

    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    ).to("cpu")

    with torch.no_grad():
        # 🔧 FIX: Handle dict output
        outputs = model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )

        # Extract logits from dict if needed
        if isinstance(outputs, dict):
            logits = outputs['logits']
        else:
            logits = outputs

        probs = torch.sigmoid(logits).cpu().numpy()
        batch_preds = (probs >= 0.5).astype(int).tolist()

    preds.extend(batch_preds)
    probs_list.extend(probs.tolist())

Predicting:   0%|          | 0/12500 [00:00<?, ?it/s]

KeyboardInterrupt: 